# 1 - Check Server Capabilities
===============================================

Complete hardware diagnostics for running LLMs locally, plus the decision
step of the extraction pipeline: 
- the final section matches the detected memory budget against the candidate models and freezes the runnable list
to ``config/runnable_models.json``, which ``3_2_download_models.ipynb`` and ``run_extraction.py`` consume.


In [ ]:
import json
import os
import platform
import shutil
import subprocess
from pathlib import Path

In [ ]:
GREEN  = "\033[92m"
YELLOW = "\033[93m"
RED    = "\033[91m"
BLUE   = "\033[94m"
BOLD   = "\033[1m"
RESET  = "\033[0m"

def ok(msg):      print(f"  {GREEN}v{RESET}  {msg}")
def warn(msg):    print(f"  {YELLOW}!{RESET}  {msg}")
def fail(msg):    print(f"  {RED}x{RESET}  {msg}")
def info(msg):    print(f"     {msg}")
def section(msg): print(f"\n{BOLD}{BLUE}{'-'*60}{RESET}\n{BOLD} {msg}{RESET}\n{'-'*60}")

In [ ]:
def run(cmd):
    """Run a shell command and return stdout, or '' on failure."""
    try:
        return subprocess.check_output(cmd, shell=True, stderr=subprocess.DEVNULL,
                                       text=True).strip()
    except Exception:
        return ""

**1. Operating system**

In [ ]:
section("Operating system")
info(f"OS      : {platform.system()} {platform.release()}")
info(f"Version : {platform.version()[:80]}")
info(f"Machine : {platform.machine()}")
info(f"Node    : {platform.node()}")

distro = run("cat /etc/os-release | grep PRETTY_NAME | cut -d= -f2 | tr -d '\"'")
if distro:
    info(f"Distro  : {distro}")

uptime = run("uptime -p 2>/dev/null || uptime")
if uptime:
    info(f"Uptime  : {uptime}")

**2. CPU**

In [ ]:
section("CPU")

try:
    import psutil
    cpu_count_logical  = psutil.cpu_count(logical=True)
    cpu_count_physical = psutil.cpu_count(logical=False)
    cpu_freq = psutil.cpu_freq()
    cpu_percent = psutil.cpu_percent(interval=1)

    info(f"Physical cores : {cpu_count_physical}")
    info(f"Logical threads: {cpu_count_logical}")
    if cpu_freq:
        info(f"Frequency      : {cpu_freq.current:.0f} MHz  (max {cpu_freq.max:.0f} MHz)")
    info(f"Current usage  : {cpu_percent:.1f}%")
    ok("psutil available — detailed CPU data")

except ImportError:
    warn("psutil not installed. Falling back to /proc/cpuinfo.")
    model = run("grep 'model name' /proc/cpuinfo | head -1 | cut -d: -f2").strip()
    cores = run("nproc")
    mhz   = run("grep 'cpu MHz' /proc/cpuinfo | head -1 | cut -d: -f2").strip()
    if model: info(f"Model  : {model}")
    if cores: info(f"Threads: {cores}")
    if mhz:   info(f"MHz    : {mhz}")

lscpu = run("lscpu | grep -E 'Architecture|Vendor|Socket|Core|Thread'")
if lscpu:
    for line in lscpu.splitlines():
        info(line.strip())

**3. RAM**

In [ ]:
section("RAM — system memory")

ram_available_gb = 0.0

try:
    import psutil
    mem = psutil.virtual_memory()
    swap = psutil.swap_memory()

    total_gb = mem.total / 1024**3
    ram_available_gb = mem.available / 1024**3
    used_gb  = mem.used / 1024**3
    swap_gb  = swap.total / 1024**3

    info(f"Total     : {total_gb:.1f} GB")
    info(f"Used      : {used_gb:.1f} GB  ({mem.percent:.0f}%)")
    info(f"Available : {ram_available_gb:.1f} GB ({100 - mem.percent:.0f}%)")
    info(f"Swap      : {swap_gb:.1f} GB")

    if ram_available_gb >= 64:   ok(f"{ram_available_gb:.0f} GB free — enough for large models (70B+)")
    elif ram_available_gb >= 32: ok(f"{ram_available_gb:.0f} GB free — enough for mid-size models (~30B)")
    elif ram_available_gb >= 16: warn(f"{ram_available_gb:.0f} GB free — enough for small models (7B-14B)")
    else:                        warn(f"{ram_available_gb:.0f} GB free — limited; use aggressive quantisation")

except ImportError:
    mem_raw = run("grep -E 'MemTotal|MemAvailable|SwapTotal' /proc/meminfo")
    for line in mem_raw.splitlines():
        info(line)
    try:
        avail_kb = int(run("grep MemAvailable /proc/meminfo").split()[1])
        ram_available_gb = avail_kb / 1024**2
    except Exception:
        pass

**4. GPU**

In [ ]:
section("GPU — graphics card")

nvidia_smi = shutil.which("nvidia-smi")

if nvidia_smi:
    ok("nvidia-smi found — NVIDIA GPU detected")

    # Per-GPU information
    query = run(
        "nvidia-smi --query-gpu=index,name,driver_version,memory.total,"
        "memory.used,memory.free,temperature.gpu,utilization.gpu,"
        "utilization.memory,power.draw,power.limit "
        "--format=csv,noheader,nounits"
    )

    gpus = []
    if query:
        for line in query.splitlines():
            parts = [p.strip() for p in line.split(",")]
            if len(parts) >= 11:
                idx, name, driver, mem_total, mem_used, mem_free, temp, util_gpu, util_mem, power, power_max = parts[:11]
                gpus.append({
                    "idx": idx, "name": name, "driver": driver,
                    "mem_total": float(mem_total), "mem_used": float(mem_used),
                    "mem_free": float(mem_free), "temp": temp,
                    "util_gpu": util_gpu, "util_mem": util_mem,
                    "power": power, "power_max": power_max,
                })
                print()
                info(f"GPU #{idx} : {name}")
                info(f"Driver  : {driver}")
                info(f"VRAM    : {float(mem_total)/1024:.1f} GB total  |  {float(mem_free)/1024:.1f} GB free  |  {float(mem_used)/1024:.1f} GB used")
                info(f"Temp    : {temp} C  |  GPU: {util_gpu}%  |  Memory: {util_mem}%")
                try:
                    info(f"Power   : {float(power):.0f} W / {float(power_max):.0f} W")
                except Exception:
                    pass

    # CUDA
    cuda_ver = run("nvidia-smi | grep 'CUDA Version' | awk '{print $NF}'")
    if cuda_ver:
        info(f"\nCUDA version (driver): {cuda_ver}")

    nvcc = run("nvcc --version | grep 'release'")
    if nvcc:
        info(f"nvcc (toolkit)       : {nvcc.strip()}")
    else:
        warn("nvcc not found on PATH (CUDA toolkit may not be installed)")

else:
    rocm = run("rocm-smi 2>/dev/null | head -5")
    if rocm:
        warn("No NVIDIA driver, but ROCm (AMD) detected:")
        info(rocm)
    else:
        lspci_gpu = run("lspci 2>/dev/null | grep -iE 'vga|3d|display'")
        if lspci_gpu:
            warn("No NVIDIA/ROCm driver, but display hardware detected:")
            for l in lspci_gpu.splitlines():
                info(l)
        else:
            fail("No GPU detected (or insufficient permissions)")
        warn("You will be limited to CPU inference (very slow for large models)")

**5. PyTorch and CUDA**

In [ ]:
section("PyTorch and CUDA")

try:
    import torch
    info(f"PyTorch version : {torch.__version__}")
    if torch.cuda.is_available():
        ok("CUDA available via PyTorch")
        info(f"Devices         : {torch.cuda.device_count()} GPU(s)")
        for i in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(i)
            vram_total = props.total_memory / 1024**3
            info(f"  GPU {i}: {props.name}  —  {vram_total:.1f} GB VRAM")
    else:
        warn("CUDA not available via PyTorch (CPU only)")
except ImportError:
    warn("PyTorch not installed. Install with: pip install torch")

**6. Storage**

In [ ]:
section("Storage")

df = run("df -h / 2>/dev/null | tail -1")
if df:
    parts = df.split()
    if len(parts) >= 5:
        info(f"Disk (/)  : {parts[1]} total  |  {parts[2]} used  |  {parts[3]} free  ({parts[4]})")
        try:
            pct = int(parts[4].replace("%", ""))
            if pct > 90:   warn("Disk almost full! LLM models need space.")
            elif pct > 75: warn(f"Disk {pct}% used — check free space before downloading models.")
            else:          ok(f"Disk {pct}% used.")
        except Exception:
            pass

# Space in common model directories
for folder in ["~/.ollama", "~/.cache/huggingface", "~/models"]:
    p = os.path.expanduser(folder)
    if os.path.exists(p):
        size = run(f"du -sh {p} 2>/dev/null | cut -f1")
        if size:
            info(f"{folder:<30}: {size} used")

**7. Installed LLM tooling**

In [ ]:
section("Installed LLM tooling")

tools = {
    "ollama"  : "Ollama (local model engine)",
    "python3" : "Python 3",
    "pip3"    : "pip (Python package manager)",
    "git"     : "Git",
    "wget"    : "wget",
    "curl"    : "curl",
    "conda"   : "Conda / Mamba",
    "docker"  : "Docker",
}

for cmd, name in tools.items():
    path = shutil.which(cmd)
    if path:
        version = run(f"{cmd} --version 2>/dev/null | head -1")
        ok(f"{name:<35} {version[:40] if version else ''}")
    else:
        info(f"  — {name} not found")

# Relevant Python packages
section("Python packages for LLMs")
packages = ["torch", "transformers", "llama_cpp", "ollama", "psutil",
            "accelerate", "bitsandbytes", "huggingface_hub", "vllm"]

for pkg in packages:
    try:
        mod = __import__(pkg.replace("-", "_"))
        ver = getattr(mod, "__version__", "?")
        ok(f"{pkg:<25} v{ver}")
    except ImportError:
        info(f"  — {pkg:<25} not installed  (pip install {pkg})")

**8. Final recommendations**

In [ ]:
section("Recommendations for LLMs")

# Collect the numbers the recommendation is based on
vram_free_gb = 0.0
n_gpus = 0

try:
    import torch
    if torch.cuda.is_available():
        n_gpus = torch.cuda.device_count()
        for i in range(n_gpus):
            props = torch.cuda.get_device_properties(i)
            vram_free_gb += props.total_memory / 1024**3
except Exception:
    pass

if vram_free_gb == 0 and nvidia_smi:
    raw = run("nvidia-smi --query-gpu=memory.free --format=csv,noheader,nounits")
    try:
        vram_free_gb = sum(float(x) for x in raw.splitlines()) / 1024
    except Exception:
        pass

print()
if vram_free_gb >= 80:
    ok(f"{vram_free_gb:.0f} GB VRAM — you can run 70B-class models (e.g. llama3.1:70b)")
elif vram_free_gb >= 40:
    ok(f"{vram_free_gb:.0f} GB VRAM — you can run 30B-class models (e.g. qwen2.5:32b)")
elif vram_free_gb >= 20:
    ok(f"{vram_free_gb:.0f} GB VRAM — you can run 14B-27B models in Q4 (e.g. medgemma:27b Q4)")
elif vram_free_gb >= 12:
    ok(f"{vram_free_gb:.0f} GB VRAM — you can run 14B-class models (e.g. qwen2.5:14b, phi4)")
elif vram_free_gb >= 6:
    ok(f"{vram_free_gb:.0f} GB VRAM — you can run 7B-9B models in Q4 (e.g. llama3.1:8b, gemma2:9b)")
elif vram_free_gb >= 4:
    ok(f"{vram_free_gb:.0f} GB VRAM — you can run 3B-7B models in Q4")
elif vram_free_gb > 0:
    warn(f"{vram_free_gb:.0f} GB VRAM — very limited; use 1B-3B models or CPU")
else:
    warn("No VRAM detected — CPU-only inference (slow)")
    if ram_available_gb >= 32:
        info(f"Available RAM: {ram_available_gb:.0f} GB — 7B-13B models are feasible on CPU (slow)")
    elif ram_available_gb >= 16:
        info(f"Available RAM: {ram_available_gb:.0f} GB — 7B models are feasible on CPU")
    else:
        info(f"Available RAM: {ram_available_gb:.0f} GB — stick to 3B models")

**9. Freeze the runnable model list for the pipeline**

This is the decision step: the detected memory budget is matched against the
experiment's candidate models, and the runnable list is written to
`config/runnable_models.json`. Downstream, `download_models.py` pulls exactly
this list and `run_extraction.py` uses it as the default `--models` value —
so server capability, not a hard-coded list, defines the experiment.

In [ ]:
# The candidate models and the output path live in config.py — the single
# source of truth for the experiment. Editing which models to compare is done
# there, not here.
from clinical_notes_extraction.config import CANDIDATE_MODELS, RUNNABLE_MODELS_FILE

CPU_RAM_HEADROOM = 0.7  # on CPU-only machines, keep 30% of RAM for the OS

# Budget: prefer GPU VRAM (fast inference); otherwise available RAM with headroom.
if vram_free_gb > 0:
    budget_gb, budget_source = vram_free_gb, "GPU VRAM"
else:
    budget_gb, budget_source = ram_available_gb * CPU_RAM_HEADROOM, "available RAM x 70%"

print(f"Memory budget per model: ~{budget_gb:.1f} GB ({budget_source})\n")

runnable = []
for model, needed_gb in CANDIDATE_MODELS.items():
    if needed_gb <= budget_gb:
        ok(f"{model:<16} ~{needed_gb:.1f} GB needed -> runnable")
        runnable.append(model)
    else:
        warn(f"{model:<16} ~{needed_gb:.1f} GB needed -> too large, excluded")

RUNNABLE_MODELS_FILE.parent.mkdir(parents=True, exist_ok=True)
with open(RUNNABLE_MODELS_FILE, "w") as f:
    json.dump({
        "models": runnable,
        "memory_budget_gb": round(budget_gb, 1),
        "budget_source": budget_source,
    }, f, indent=2)

print(f"\nRunnable models written to {RUNNABLE_MODELS_FILE}: {runnable}")
print("Next step: python scripts/download_models.py")